# 15 — Astra tools: network accessibility and uncertainty

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/15_astra_network_accessibility.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability. Widget/anywidget/LexCube branches need their source extensions; the Atlas shell supplies only Pyodide.

**Provenance:** `book/notebooks/15_astra_network_accessibility.ipynb` at `777fa8965068b1e26b982162f17254ee2df7ec84`; original SHA-256 `194668d1168027e01df6040a0afa5828f57466c9254fd6f8c11ccb5ed887de90`. Original author material incorporated at repository owner request; no general license inferred. Third-party terms remain separate.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 15 — Astra tools: network accessibility and uncertainty

[Launch in JupyterLite](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=15_astra_network_accessibility.ipynb)

Compute shortest travel times across a synthetic town, close a bridge, and inspect the resulting network, map and sensitivity curves. Give Astra a bounded analysis tool that summarizes computed routes. **Time:** about 30 minutes. **Prerequisites:** Python dictionaries; no graph library required.

**Runtime:** browser Python for the analysis; hosted GeoLibre for maps; optional CPython runner for Astra. The default run makes no model calls. The first run needs network access for packages, GeoLibre and interactive Plotly scripts. Static plots and local analysis do not depend on data APIs.

**Learning cycle:** predict → run → compare views → explain → test a different assumption.

## Start here: the question behind the code

**How does one closure change access to a destination?**

**What you will learn:** connect → reroute → compare → test. The numbered reading guides below follow the actual code cells. Read a guide, predict the output, then run its cell with **Shift+Enter**. Run from top to bottom; changing an upstream value requires rerunning the cells that use it.

**GeoAI connection:** Graph analysis plus optional bounded model tools; shortest paths are computed by an explicit algorithm.

**Why this is useful:** A small controlled network reveals how connectivity and assumptions affect an apparently simple access percentage.

**Where the work happens:** Network construction, shortest paths, coverage and tool replay run in Pyodide. Live model calls use the optional external runner.

![A bridge changes the reachable world. Represent travel possibilities as nodes, edges and minute costs. Remove one bridge and recompute routes to the same destination. Show who gains travel time and how a threshold changes covered-node counts. Vary speed assumptions before asking an optional model to interpret computed evidence.](assets/learning/15/infographic.svg)

### Watch the idea unfold

[[Source image continuation: Preview: A bridge changes the reachable world — 32-second captioned walkthrough](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/assets/learning/15/poster.png)](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/15/watch.html)

[▶ Watch the captioned mini-video and read its transcript](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/15/watch.html) · [Launch this notebook](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=15_astra_network_accessibility.ipynb) · [Read this Book chapter](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/notebooks/15_astra_network_accessibility.html) · [Python and browser learning guide](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/learning-guide.html)

The video is a silent four-step teaching storyboard with player controls and captions. Open the companion page to play, pause, seek or download it. The local copy is bundled under `assets/learning/15/`; downloaded media can be played offline. The infographic and the text below cover the same ideas without a video player.

**Video transcript:**

1. **Connect (0–8 s):** Represent travel possibilities as nodes, edges and minute costs. Code landmark: `edges.append((a, b, 2.0))`.

2. **Reroute (8–16 s):** Remove one bridge and recompute routes to the same destination. Code landmark: `shortest_times(closed=True)`.

3. **Compare (16–24 s):** Show who gains travel time and how a threshold changes covered-node counts. Code landmark: `routes['delay']`.

4. **Test (24–32 s):** Vary speed assumptions before asking an optional model to interpret computed evidence. Code landmark: `routes[column] * 1.2`.


## See the connections

![Accessibility: networks and counterfactuals: Network: Edges carry travel time, not straight-line distance; Scenario: Closing a bridge changes shortest routes; Astra tools: A bounded tool summarizes computed evidence; Check: Reachable nodes are not population coverage](assets/15_mindmap.svg)

### Code guide 1 — Prepare the browser toolbox

**Read the Python:** `import` makes a package available to Python. `sys.platform` identifies the runtime; the indented `if` block runs only in Pyodide. `await` waits for package loading. `LiteMap as Map` gives the browser-compatible map class a shorter name. Installing a package and importing its functions are separate steps.

**Look for:** Imports complete before you run the next cell. The first run can take longer while packages download.

**Pause and try:** Find the alias `Map`. If a later cell says a name is undefined, rerun this setup cell first.


In [ ]:
import sys
if sys.platform == "emscripten":
    from pyodide import loadPackage
    await loadPackage(["numpy", "pandas", "matplotlib", "scipy", "pillow"])
    import micropip
    await micropip.install(["geolibre==3.0.0", "plotly==6.5.0"])
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from geolibre_lite import LiteMap as Map
from cognition import style_plots, COLORS, save_json
from astra_lab import make_request, response_text
style_plots()

## 1. Model the network, including the barrier

The 6 × 6 grid is invented. Each street segment takes two minutes; a river splits the grid between columns 2 and 3. Only two bridges cross it. The destination is node 0 (southwest). Closing one bridge is a counterfactual, not a real emergency scenario.

Predict which side of the river will gain the most travel time. Straight-line distance will not capture the detour.

### Code guide 2 — Follow a shortest-path algorithm

**Read the Python:** The grid creates nodes and two-minute edges, with only two crossings of the river. The function builds neighbor lists, starts node 0 at time zero and uses a priority queue to visit the cheapest known route next. A shorter candidate replaces the stored time. Skipping one bridge changes connectivity; assertions check hand-traced examples.

**Look for:** Baseline and closed-bridge times, including 8 versus 20 minutes at node 9.

**Pause and try:** Trace node 9's detour on the diagram and explain why straight-line distance misses it.


In [ ]:
import heapq
from cognition import local_lonlat
n=6
positions={r*n+c:(c*300,r*300) for r in range(n) for c in range(n)}
edges=[]
for r in range(n):
    for c in range(n):
        node=r*n+c
        if r<n-1: edges.append((node,node+n,2.0))
        if c<n-1 and (c!=2 or r in (1,4)): edges.append((node,node+1,2.0))
closed_bridge=(8,9)
def shortest_times(closed=False):
    graph={node:[] for node in positions}
    for a,b,cost in edges:
        if closed and (a,b)==closed_bridge: continue
        graph[a].append((b,cost)); graph[b].append((a,cost))
    best={node:float('inf') for node in positions}; best[0]=0
    queue=[(0,0)]
    while queue:
        time,node=heapq.heappop(queue)
        if time!=best[node]: continue
        for neighbor,cost in graph[node]:
            candidate=time+cost
            if candidate<best[neighbor]:
                best[neighbor]=candidate; heapq.heappush(queue,(candidate,neighbor))
    return best
baseline=shortest_times()
closure=shortest_times(closed=True)
routes=pd.DataFrame([{"node":node,"baseline":baseline[node],"bridge_closed":closure[node]} for node in positions])
routes['delay']=routes.bridge_closed-routes.baseline
display(routes.sort_values('delay',ascending=False).head())
assert baseline[0] == closure[0] == 0
assert all(closure[k] >= baseline[k] for k in positions)
assert baseline[9] == 8 and closure[9] == 20  # independent hand-traced paths

### Code guide 3 — Compare route maps with coverage curves

**Read the Python:** Both maps share a color scale, so the same color means the same number of minutes. For each threshold, `.le(t).mean()` computes the fraction of nodes reachable within that time. The step chart reflects discrete nodes.

**Look for:** Before/after network maps and two accessibility curves, with node 0 marked as the destination.

**Pause and try:** At 12 minutes, count covered nodes. Explain why this is not yet a percentage of residents.


In [ ]:
fig,panels=plt.subplot_mosaic([["baseline","closed"],["curve","curve"]],figsize=(10,8),layout="constrained")
axes=[panels[k] for k in ["baseline","closed","curve"]]
for ax,column,title in zip(axes[:2],['baseline','bridge_closed'],['Baseline travel time','Bridge closed: same destination']):
    ax.axvline(.75,color="#7dd3fc",lw=12,alpha=.5,zorder=0)
    for a,b,cost in edges:
        pa,pb=positions[a],positions[b]
        is_closed=column=='bridge_closed' and (a,b)==closed_bridge
        ax.plot([pa[0]/1000,pb[0]/1000],[pa[1]/1000,pb[1]/1000],
                color="#dc2626" if is_closed else "#94a3b8",ls="--" if is_closed else "-",zorder=1)
    pts=ax.scatter([p[0]/1000 for p in positions.values()],[p[1]/1000 for p in positions.values()],
                   c=routes[column],vmin=0,vmax=routes[['baseline','bridge_closed']].to_numpy().max(),cmap="viridis",s=70,zorder=2)
    ax.scatter(0,0,marker="*",s=200,c="#d97706",zorder=3)
    ax.set(xlabel="Local east (km)",ylabel="Local north (km)",title=title,aspect="equal")
    fig.colorbar(pts,ax=ax,label="Minutes to node 0")
thresholds=np.arange(0,33,2)
for column,label in [('baseline','Baseline'),('bridge_closed','Bridge closed')]:
    axes[2].step(thresholds,[100*routes[column].le(t).mean() for t in thresholds],where="post",label=label)
axes[2].set(xlabel="Travel-time threshold (minutes)",ylabel="Nodes reachable (%)",title="Accessibility depends on the cutoff")
axes[2].legend()
plt.show()

## 2. Compare geometry with the graph

GeoJSON carries the same nodes and edges. Point size encodes the extra minutes caused by closure, with a fixed minimum radius to keep zero-delay nodes visible. This counts nodes, not people, and the graph has no congestion or capacity constraints.

### Code guide 4 — Map the network with explicit attributes

**Read the Python:** Each edge becomes a LineString and each node becomes a marker. `local_lonlat` approximately anchors the invented grid on a basemap. Marker radius adds a fixed minimum to half the delay, keeping zero-delay nodes visible.

**Look for:** A synthetic street map whose point properties contain baseline, closure and delay minutes.

**Pause and try:** Compare a large marker's attributes with the table to decode its size.


In [ ]:
network_fc={"type":"FeatureCollection","features":[]}
for a,b,cost in edges:
    network_fc['features'].append({"type":"Feature","properties":{"minutes":cost,"closed":(a,b)==closed_bridge},
        "geometry":{"type":"LineString","coordinates":[local_lonlat(*positions[a]),local_lonlat(*positions[b])]}})
network_map=Map(center=(-122.291,37.907),zoom=13,height="520px")
network_map.add_geojson(network_fc,name="Synthetic streets",strokeColor="#64748b",strokeWidth=2)
for row in routes.itertuples():
    lon,lat=local_lonlat(*positions[row.node])
    network_map.add_marker(lon,lat,name=f"Node {row.node}: +{row.delay:.0f} min",
        properties={"node":row.node,"baseline_min":row.baseline,"closed_min":row.bridge_closed,"delay_min":row.delay},
        circleRadius=4+row.delay/2,fillColor="#d97706" if row.delay>0 else "#2563eb")
display(network_map)

## 3. Quantify sensitivity instead of inventing confidence

The band below varies all edge travel times by ±20%. It is a **scenario envelope**, not a statistical confidence interval. The closure still changes topology; the speed factor changes weights. Use both to explain why a single coverage percentage is incomplete.

### Code guide 5 — Test the speed assumption

**Read the Python:** Multiplying travel times by 0.8 and 1.2 creates faster/slower scenarios. Each is evaluated at the same thresholds. `fill_between` shades their envelope; these two assumed scenarios do not estimate a statistical confidence interval.

**Look for:** Coverage curves with bands showing sensitivity to a ±20% travel-time change.

**Pause and try:** Which matters more at your chosen cutoff: the bridge closure or the assumed speed?


In [ ]:
fig,ax=plt.subplots(figsize=(10,4),layout="constrained")
for column,label,color in [('baseline','Baseline',COLORS[0]),('bridge_closed','Bridge closed',COLORS[1])]:
    fast=np.array([100*(routes[column]*.8).le(t).mean() for t in thresholds])
    slow=np.array([100*(routes[column]*1.2).le(t).mean() for t in thresholds])
    ax.fill_between(thresholds,slow,fast,step="post",alpha=.18,color=color)
    ax.step(thresholds,[100*routes[column].le(t).mean() for t in thresholds],where="post",label=label,color=color)
ax.set(xlabel="Minutes",ylabel="Reachable nodes (%)",title="±20% travel-time scenarios; not a confidence interval")
ax.legend()
plt.show()

## 4. Give Astra a real bounded tool

`summarize_accessibility` can only select one of the two scenarios and a threshold from 0 to 120 minutes. Python computes the result from exported routes. The replay below uses **hand-authored arguments**, not a model call. Try an unknown tool name and check that dispatch rejects it.

For live use, the CPython runner sends the tool definition to Astra, validates its arguments, returns the calculated result using `function_call_output`, and preserves response reasoning items between rounds. It never grants shell access or executes generated code.

### Code guide 6 — Expose a bounded calculation as a tool

**Read the Python:** The replay passes a named scenario and threshold to a permitted local function, which calculates from the route evidence. The request advertises that tool for optional model use. Separate files retain the full evidence and map recipe for inspection.

**Look for:** A computed replay result and exported files; these arguments were hand-authored, not chosen by a live model.

**Pause and try:** Compare the replay fraction with your own count at 12 minutes.


In [ ]:
from astra_lab import ACCESS_TOOL, dispatch_tool
evidence={"evidence_id":"routes","source":"synthetic 6x6 network","units":"minutes",
          "destination_node":0,"closed_bridge":list(closed_bridge),"routes":routes.to_dict('records')}
replay=dispatch_tool("summarize_accessibility",{"scenario":"bridge_closed","threshold_minutes":12},evidence)
display(replay)
request=make_request("Use summarize_accessibility for BOTH baseline and bridge_closed at 12 minutes. Compare the returned fractions. Cite routes; explain why node coverage is not population coverage and why the speed band is not a confidence interval.",
                     {k:v for k,v in evidence.items() if k!='routes'})
request['tools']=[ACCESS_TOOL]
save_json("exports/accessibility_request.json",request)
save_json("exports/accessibility_evidence.json",evidence)
save_json("exports/accessibility.geolibre.json",network_map.to_project())

## Optional: run this request with GPT-6 Astra

All calculations above run locally. The exported request has **not been sent** and any example plan is **hand-authored**, not a recorded model response. To obtain a live response:

1. Download the generated request from JupyterLite's file browser (right-click → Download).
2. In a full CPython terminal in this repository, set `OPENAI_API_KEY` in that terminal's environment. Never put the key in a notebook, output, or the static site.
3. Preview with `python scripts/run_astra.py PATH_TO_REQUEST.json`. Send deliberately with `--send --output astra_response.json`. This uses your API account and can incur charges.
4. Upload `astra_response.json` to the notebook folder, set the import path in the next cell, and rerun it. Inspect its evidence claims before using them.

For notebook 15, also download `accessibility_evidence.json` and pass `--evidence PATH_TO_EVIDENCE.json`. The runner only dispatches the one allowlisted local tool, with a bounded number of rounds.

The static JupyterLite site does not host this runner or proxy requests. Model access depends on your API project. Image reading is an aid to interpretation; geometry, distances and metrics come from code and explicit metadata.

### Code guide 7 — Inspect an optional model response

**Read the Python:** `None` means no response file was supplied. Only a real path enters the `if` branch; `json.loads` converts saved JSON text into Python objects. The `else` branch keeps the default lesson usable without an API call. Follow the preceding export/import instructions for live use.

**Look for:** The default run prints an offline message. An imported answer is evidence to inspect, not a new measurement.

**Pause and try:** Write your own explanation first, then check any imported response against the computed values and units.


In [ ]:
RESPONSE_PATH=None
if RESPONSE_PATH:
    display(json.loads(response_text(json.loads(Path(RESPONSE_PATH).read_text()))))
else:
    print("Replay only. No Astra tool calls have been made.")

## 5. Evaluate a spatial explanation

Score a live answer against these checks: does it cite `routes`, use both computed fractions, identify the cutoff, distinguish nodes from people, and label the speed band correctly? A fluent answer can fail any of these checks.

**Transfer:** replace the fixture with a licensed road graph and validated edge costs in full Python. Join population only after choosing a defensible allocation method, account for disconnected nodes, and document the network date. Return GeoJSON predictions to GeoLibre and use Astra to review evidence and propose testable follow-up questions.

**Experiment:** close both bridges. Some nodes become unreachable; encode those times as JSON `null`, never `Infinity`, before using the provided tool. Explain why a straight-line buffer would miss the disconnection.

## Data & software citations

- Teaching fixtures: deterministic synthetic arrays and networks authored in this repository. Values, scenes, facilities and outcomes are invented; coordinates only anchor the display. No actual people, environmental measurements, or service areas are represented.
- [GeoLibre](https://geolibre.app/) and [GeoLibre map controls](https://geolibre.app/user-guide/map-controls/); `geolibre==3.0.0` project builders through `geolibre_lite.LiteMap`.
- [GeoAI](https://opengeoai.org/): full Python/GPU continuation for imagery models. These browser labs implement workflow patterns; they do not install or run `geoai-py`.
- [GPT-6 Astra model](https://developers.openai.com/api/docs/models/gpt-6-astra), [Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs), [image inputs](https://developers.openai.com/api/docs/guides/images-vision), and [function calling](https://developers.openai.com/api/docs/guides/function-calling). API examples checked 2026-10-02.
- [Pyodide packages](https://pyodide.org/en/stable/usage/packages-in-pyodide.html), [Matplotlib](https://matplotlib.org/stable/), [Plotly Python](https://plotly.com/python/).
- GeoLibre basemap tiles retain their provider attribution, including [OpenStreetMap contributors](https://www.openstreetmap.org/copyright) where applicable.

See `DATA_SOURCES.md` for the source register and runtime boundaries.

## Make sense of the result

![The clinic is nearby, so everyone can reach it. A river and a closed bridge can force a long detour. I will measure network travel time, then consider people.](assets/learning/15/cartoon.svg)

**Cartoon transcript:** The clinic is nearby, so everyone can reach it. → A river and a closed bridge can force a long detour. → I will measure network travel time, then consider people.

**Explain it to a colleague:** Does 80% node coverage mean 80% population coverage?

<details><summary>Check your reasoning</summary>

No. Node coverage treats nodes equally. Population coverage needs validated population allocation and a defined denominator.

</details>

Write three sentences: **I used…** (source and units), **the code computed…** (method), and **I can conclude…** (plus a remaining question). Point to the map, chart or table that supports each claim.

All new teaching illustrations are original schematics. Their marks and fictional dialogue are explanatory, not observations, fitted predictions or captured AI conversations. The existing citation cells identify the analytical data.


## GeoLibre creator and software attribution

GeoLibre is developed by **Qiusheng Wu and contributors**. Recommended citation: Wu, Q. (2026). *GeoLibre: A lightweight, cloud-native GIS platform for visualizing, exploring, and analyzing geospatial data.* Zenodo. [doi:10.5281/zenodo.20785400](https://doi.org/10.5281/zenodo.20785400). See [the official citation guidance](https://geolibre.app/citation/) and [source and MIT license](https://github.com/opengeos/GeoLibre). These browser lessons pin the Python builders to `geolibre==3.0.0`; the independently hosted viewer may change. This repository supplies the JupyterLite adapter and teaching materials, not the upstream GIS. Keep the original dataset, demo-author and basemap credits when sharing a result.
